# 01 · GA4 Funnel EDA — Where Does the Pipeline Leak?

**Analytical question.** On the GA4 obfuscated e-commerce sample, which of the five transitions — `session_start → view_item → add_to_cart → begin_checkout → purchase` — sheds the most users? Treating that stage as the target of one disciplined A/B test, what does a reasonable lift translate to at portfolio scale, in rupiah?

**Why this matters.** FMCG funnels live and die on where the cohort narrows. The biggest drop-off identifies the bottleneck, and that is where experiments should be deployed first. This notebook finds the bottleneck on the cleanest public e-commerce dataset available, then prices the treatment in IDR using the baseline documented in this repo.

**Sessionization choice.** GA4's native `ga_session_id` parameter is used over a recomputed 30-minute inactivity window. Full rationale in the header of `sql/01_funnel_decomposition.sql`.

**Prerequisites.** `uv sync`, `gcloud auth application-default login`, and a populated `.env` with `GCP_PROJECT_ID`.

---

## 1 · Setup

BigQuery work flows through `smokefreelab.data.bigquery` — one client wrapper with parameterization, cost-guarding, and `.env` discovery. Visual theming flows through `smokefreelab.analytics.viz` so every chart in the project inherits the same editorial palette and layout.

In [1]:
from __future__ import annotations

import pandas as pd
import plotly.graph_objects as go

from smokefreelab.analytics.viz import (
    COLOR_ACCENT,
    COLOR_MUTED,
    COLOR_POSITIVE,
    COLOR_PRIMARY,
    FUNNEL_PALETTE,
    add_insight_annotation,
    apply_sfl_theme,
    format_rupiah,
)
from smokefreelab.data.bigquery import (
    BQConfig,
    estimate_query_bytes,
    load_sql,
    run_query,
)

pd.set_option("display.float_format", "{:,.4f}".format)

cfg = BQConfig()
print(f"Billing project    : {cfg.gcp_project_id}")
print(f"Dataset location   : {cfg.bq_location}")
print(f"GA4 sample dataset : {cfg.ga4_project}.{cfg.ga4_dataset}")

Billing project    : streamring-research
Dataset location   : US
GA4 sample dataset : bigquery-public-data.ga4_obfuscated_sample_ecommerce


## 2 · Cost check — dry run before we bill anything

Sandbox tier is 1 TB scanned per month. Every query estimates bytes first; a misfiltered `_TABLE_SUFFIX` can burn the monthly budget in one run. Target for this query: under 2 GB — ~0.2% of the monthly budget.

In [2]:
sql = load_sql("01_funnel_decomposition")
params = {"start_date": "20201101", "end_date": "20210131"}

bytes_scanned = estimate_query_bytes(sql, params=params)
gb_scanned = bytes_scanned / 1e9
print(f"Dry-run scan estimate : {bytes_scanned:,} bytes ({gb_scanned:.2f} GB)")
assert gb_scanned < 2.0, (
    f"Query would scan {gb_scanned:.2f} GB — investigate before running."
)

Dry-run scan estimate : 1,029,558,211 bytes (1.03 GB)


## 3 · Execute the funnel query

In [3]:
funnel_daily = run_query(sql, params=params)
funnel_daily["event_date"] = pd.to_datetime(funnel_daily["event_date"])

print(f"Rows returned : {len(funnel_daily):,}")
print(
    f"Window        : {funnel_daily['event_date'].min().date()} "
    f"→ {funnel_daily['event_date'].max().date()}"
)
funnel_daily.head(10)

Rows returned : 442
Window        : 2020-11-01 → 2021-01-31


,event_date,stage,users,sessions,cvr_vs_prior_stage,cvr_vs_entry
0,2020-11-01,session_start,2339,2594,NaN,1.0000
1,2020-11-01,view_item,516,537,0.2206,0.2206
2,2020-11-01,add_to_cart,1,1,0.0019,0.0004
3,2020-11-01,begin_checkout,91,95,91.0000,0.0389
4,2020-11-01,purchase,13,13,0.1429,0.0056
5,2020-11-02,session_start,3227,3682,NaN,1.0000
6,2020-11-02,view_item,782,819,0.2423,0.2423
7,2020-11-02,begin_checkout,127,130,0.1624,0.0394
8,2020-11-02,purchase,39,40,0.3071,0.0121
9,2020-11-03,session_start,4604,5281,NaN,1.0000


## 4 · Collapse to a single-window funnel

The daily series is retained for the heatmap in §8; the headline visuals need one row per stage. `cvr_vs_prior` is the fraction retained from the stage above; `cvr_vs_entry` is the cumulative retention from `session_start`.

In [4]:
STAGE_ORDER = [
    "session_start",
    "view_item",
    "add_to_cart",
    "begin_checkout",
    "purchase",
]
STAGE_LABELS = {
    "session_start":  "Session start",
    "view_item":      "View item",
    "add_to_cart":    "Add to cart",
    "begin_checkout": "Begin checkout",
    "purchase":       "Purchase",
}

funnel_total = (
    funnel_daily.groupby("stage", as_index=False)[["users", "sessions"]]
    .sum()
    .assign(stage_order=lambda d: d["stage"].map(STAGE_ORDER.index))
    .sort_values("stage_order")
    .reset_index(drop=True)
)

entry_users = int(
    funnel_total.loc[funnel_total["stage"] == "session_start", "users"].iat[0]
)
funnel_total["cvr_vs_prior"]     = funnel_total["users"] / funnel_total["users"].shift(1)
funnel_total["cvr_vs_entry"]     = funnel_total["users"] / entry_users
funnel_total["dropoff_vs_prior"] = 1 - funnel_total["cvr_vs_prior"]
funnel_total["stage_label"]      = funnel_total["stage"].map(STAGE_LABELS)

funnel_total[[
    "stage_label", "users", "sessions",
    "cvr_vs_prior", "cvr_vs_entry", "dropoff_vs_prior",
]]

,stage_label,users,sessions,cvr_vs_prior,cvr_vs_entry,dropoff_vs_prior
0,Session start,314367,354857,<NA>,1.0000,<NA>
1,View item,72384,77020,0.2303,0.2303,0.7697
2,Add to cart,14360,15188,0.1984,0.0457,0.8016
3,Begin checkout,10700,11106,0.7451,0.0340,0.2549
4,Purchase,4791,4848,0.4478,0.0152,0.5522


## 5 · Headline numbers

The one-sentence version of this notebook — printed before the visuals, so the manager who only skims gets the answer.

In [5]:
total_sessions = int(
    funnel_total.loc[funnel_total["stage"] == "session_start", "sessions"].iat[0]
)
total_purchases = int(
    funnel_total.loc[funnel_total["stage"] == "purchase", "users"].iat[0]
)
overall_cvr = total_purchases / entry_users

worst_leak_row = (
    funnel_total.iloc[1:]
    .sort_values("dropoff_vs_prior", ascending=False)
    .iloc[0]
)
worst_stage   = worst_leak_row["stage_label"]
prior_stage   = STAGE_LABELS[
    STAGE_ORDER[STAGE_ORDER.index(worst_leak_row["stage"]) - 1]
]
worst_dropoff = float(worst_leak_row["dropoff_vs_prior"])

print(f"Sessions analysed       : {total_sessions:>10,}")
print(f"Unique entrants         : {entry_users:>10,}")
print(f"Completed purchases     : {total_purchases:>10,}")
print(f"Overall CVR (entry→buy) : {overall_cvr:>10.2%}")
print(f"Worst-leak transition   : {prior_stage} → {worst_stage}  "
      f"({worst_dropoff:.0%} lost)")

Sessions analysed       :    354,857
Unique entrants         :    314,367
Completed purchases     :      4,791
Overall CVR (entry→buy) :      1.52%
Worst-leak transition   : View item → Add to cart  (80% lost)


## 6 · Visual 1 — hero funnel

The canonical chart. Width encodes absolute users; the in-bar percentage is cumulative retention from entry. One insight callout flags the biggest leak — the transition that the A/B framework notebook should target first.

In [6]:
fig_funnel = go.Figure(
    go.Funnel(
        y=funnel_total["stage_label"],
        x=funnel_total["users"],
        textposition="inside",
        textinfo="value+percent initial",
        texttemplate="<b>%{value:,.0f}</b><br>%{percentInitial}",
        textfont={"size": 14, "color": "white"},
        marker={
            "color": FUNNEL_PALETTE,
            "line": {"color": "white", "width": 1.5},
        },
        connector={"line": {"color": COLOR_MUTED, "width": 1}},
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Users: %{x:,.0f}<br>"
            "vs entry: %{percentInitial}<extra></extra>"
        ),
    )
)
fig_funnel.update_layout(title="Five-stage acquisition funnel")

window_start = funnel_daily["event_date"].min().strftime("%b %Y")
window_end   = funnel_daily["event_date"].max().strftime("%b %Y")
apply_sfl_theme(
    fig_funnel,
    height=560,
    subtitle=f"GA4 obfuscated e-commerce sample · {window_start} – {window_end}",
)
add_insight_annotation(
    fig_funnel,
    text=(
        f"<b>{worst_dropoff:.0%} drop at "
        f"{prior_stage} → {worst_stage}</b><br>"
        "Biggest single leak in the funnel —<br>"
        "the stage to A/B test first."
    ),
    x=1.01, y=0.92,
    xref="paper", yref="paper",
)
fig_funnel.show()

## 7 · Visual 2 — Sankey: retention versus drop-off at each step

Funnels count survivors. Sankey diagrams also count the dead: at each stage, the grey ribbon off the node is the lost cohort, the navy ribbon is the users who continued. The width of the first grey ribbon tells you, at a glance, that *discovery* is the commercial problem — not checkout friction.

In [7]:
stage_labels = [STAGE_LABELS[s] for s in STAGE_ORDER]
drop_labels  = [f"Lost after {lbl.lower()}" for lbl in stage_labels[:-1]]
node_labels  = stage_labels + drop_labels
node_colors  = list(FUNNEL_PALETTE) + [COLOR_MUTED] * len(drop_labels)

retain_rgba = "rgba(31, 58, 95, 0.55)"    # primary navy @ 55%
drop_rgba   = "rgba(168, 180, 192, 0.35)" # muted @ 35%

sources: list[int] = []
targets: list[int] = []
values:  list[int] = []
colors:  list[str] = []

users_by_stage = funnel_total.set_index("stage")["users"].to_dict()
for i, stage in enumerate(STAGE_ORDER[:-1]):
    next_stage = STAGE_ORDER[i + 1]
    retained = int(users_by_stage[next_stage])
    lost     = int(users_by_stage[stage] - users_by_stage[next_stage])

    sources.append(i); targets.append(i + 1); values.append(retained)
    colors.append(retain_rgba)

    sources.append(i); targets.append(len(stage_labels) + i); values.append(lost)
    colors.append(drop_rgba)

fig_sk = go.Figure(
    go.Sankey(
        arrangement="snap",
        node={
            "label":     node_labels,
            "color":     node_colors,
            "pad":       18,
            "thickness": 18,
            "line":      {"color": "white", "width": 1},
        },
        link={
            "source": sources,
            "target": targets,
            "value":  values,
            "color":  colors,
        },
    )
)
fig_sk.update_layout(title="Sankey — retention (navy) versus drop-off (grey)")
apply_sfl_theme(
    fig_sk,
    height=560,
    subtitle="Thicker grey ribbon ⇒ larger lost cohort. The first grey ribbon is the one to treat.",
)
fig_sk.show()

## 8 · Visual 3 — daily CVR heatmap

A single aggregated funnel hides calendar effects. The heatmap plots `cvr_vs_entry` by day × stage: dark columns flag high-conversion days (campaign lift, newsletter drops), light columns flag weak days (bot spikes, traffic mis-match). Horizontal banding confirms which stage structurally leaks the most regardless of day.

In [8]:
daily_cvr = (
    funnel_daily.pivot(index="event_date", columns="stage", values="users")
    .reindex(columns=STAGE_ORDER)
    .fillna(0)
)
daily_cvr_entry = daily_cvr.div(daily_cvr["session_start"], axis=0)

# Drop session_start row (always 1.0; not visually informative).
heat_matrix = daily_cvr_entry[STAGE_ORDER[1:]].T
y_labels = [STAGE_LABELS[s] for s in STAGE_ORDER[1:]]

fig_hm = go.Figure(
    go.Heatmap(
        z=heat_matrix.values,
        x=heat_matrix.columns,
        y=y_labels,
        colorscale=[
            [0.0, "#FAFAF5"],
            [0.25, "#D9DEE4"],
            [0.55, "#6B8CAE"],
            [1.0, "#1F3A5F"],
        ],
        colorbar={
            "title":        {"text": "CVR vs entry", "font": {"size": 11}},
            "tickformat":   ".0%",
            "outlinewidth": 0,
            "thickness":    14,
        },
        hovertemplate="<b>%{y}</b><br>%{x|%b %d, %Y}<br>CVR: %{z:.1%}<extra></extra>",
    )
)
fig_hm.update_layout(title="Daily conversion rate versus session_start")
apply_sfl_theme(
    fig_hm,
    height=420,
    subtitle="Rows = funnel stages (top → bottom of funnel). Dark = retained; light = leak.",
)
fig_hm.update_yaxes(autorange="reversed")
fig_hm.show()

## 9 · Visual 4 — stickiness with an e-commerce benchmark band

`DAU / MAU` is the canonical product-health metric: of the users who visited in the last 30 days, how many visited today? Social platforms run 40–60%. Transactional e-commerce typically sits 3–10%. The sample sits near the floor of that band, which is expected — purchase cadence is weeks, not minutes.

In [9]:
daily_dau = daily_cvr["session_start"].rename("dau").to_frame()
daily_dau["mau"]        = daily_dau["dau"].rolling(30, min_periods=1).sum()
daily_dau["stickiness"] = daily_dau["dau"] / daily_dau["mau"]

fig_st = go.Figure()
# Benchmark band: 3-10% typical e-commerce stickiness.
fig_st.add_hrect(
    y0=0.03, y1=0.10,
    fillcolor="rgba(79, 121, 66, 0.10)",
    line_width=0,
    annotation_text="Typical e-commerce band (3–10%)",
    annotation_position="top left",
    annotation_font_color=COLOR_POSITIVE,
    annotation_font_size=11,
)
fig_st.add_trace(
    go.Scatter(
        x=daily_dau.index,
        y=daily_dau["stickiness"],
        mode="lines",
        line={"color": COLOR_PRIMARY, "width": 2.5},
        name="DAU / MAU",
        hovertemplate="<b>%{x|%b %d, %Y}</b><br>Stickiness: %{y:.2%}<extra></extra>",
    )
)
mean_stick = daily_dau["stickiness"].mean()
fig_st.add_hline(
    y=mean_stick,
    line={"color": COLOR_ACCENT, "width": 1.5, "dash": "dot"},
    annotation_text=f"Window mean: {mean_stick:.2%}",
    annotation_position="bottom right",
    annotation_font_color=COLOR_ACCENT,
    annotation_font_size=11,
)
fig_st.update_layout(title="Stickiness (DAU / 30-day MAU)")
fig_st.update_yaxes(tickformat=".1%", rangemode="tozero")
apply_sfl_theme(
    fig_st,
    height=440,
    subtitle="Green band marks the 3–10% range typical of transactional e-commerce.",
)
fig_st.show()

print(f"Mean DAU/MAU over window : {daily_dau['stickiness'].mean():.2%}")
print(f"Median DAU/MAU           : {daily_dau['stickiness'].median():.2%}")

Mean DAU/MAU over window : 6.76%
Median DAU/MAU           : 3.86%


## 10 · Visual 5 — rupiah impact at portfolio scale

The funnel above tells us *where* to target. This chart says *how much* a successful A/B test at that stage is worth. Using the portfolio baseline — 250K monthly SFP registrants, IDR 450K LTV per activated user — each additional percentage point of CVR at the worst-leak stage is worth the annualized amount shown.

In [10]:
# Extrapolation baseline for the portfolio narrative.
MONTHLY_REGISTRANTS = 250_000
LTV_PER_USER        = 450_000  # IDR
MONTHS_PER_YEAR     = 12

lift_scenarios = [0.005, 0.01, 0.02, 0.03]  # 0.5pp, 1pp, 2pp, 3pp
annual_values = [
    lift * MONTHLY_REGISTRANTS * LTV_PER_USER * MONTHS_PER_YEAR
    for lift in lift_scenarios
]
labels = [f"+{lift * 100:.1f}pp" for lift in lift_scenarios]
bar_colors = [COLOR_MUTED, COLOR_PRIMARY, COLOR_ACCENT, COLOR_POSITIVE]

fig_idr = go.Figure(
    go.Bar(
        x=labels,
        y=annual_values,
        marker={"color": bar_colors, "line": {"color": "white", "width": 1.5}},
        text=[format_rupiah(v) for v in annual_values],
        textposition="outside",
        textfont={"size": 14, "color": COLOR_PRIMARY},
        hovertemplate="<b>Lift %{x}</b><br>Annualised CLV: %{text}<extra></extra>",
    )
)
fig_idr.update_layout(title="Annualised incremental CLV by lift scenario")
fig_idr.update_yaxes(
    tickformat=".2s",
    title_text="Annualised CLV (IDR)",
    rangemode="tozero",
)
apply_sfl_theme(
    fig_idr,
    height=460,
    subtitle=(
        f"Baseline: {MONTHLY_REGISTRANTS:,} monthly registrants "
        f"× IDR {LTV_PER_USER:,} LTV × 12 months."
    ),
)
add_insight_annotation(
    fig_idr,
    text=(
        "<b>README hero number</b><br>"
        f"+2pp ⇒ {format_rupiah(annual_values[2])}/yr<br>"
        "comes from this scenario."
    ),
    x=0.02, y=0.97,
    xref="paper", yref="paper",
    arrow=False,
)
fig_idr.show()

## 11 · Caveats — what would make these numbers wrong

1. **Obfuscation.** The sample strips `user_id`, granular `device.*`, and parts of `geo.*`. Segmentation below country x category is unreliable.
2. **Bot traffic.** Automated traffic leaks past GA4's native bot filter. A production pass should exclude events where `traffic_source.source IN ('(not set)', 'bot')` or impose a per-user event-count floor.
3. **Fixed window.** Nov 2020 - Jan 2021 over-weights Black Friday and year-end. Any annualised extrapolation is scaled to 12 months, not grossed up from the window.
4. **Skewed purchase revenue.** Heavily right-tailed. Downstream ARPU / AOV should use median + IQR, not mean.
5. **Sessionization.** `ga_session_id` inherits GA4's edge cases (cross-midnight rollover, app-resume behaviour) - industry standard, flag at interview.
6. **`session_start` over-count.** GA4 fires `session_start` on every new session including bounces, so the *absolute* top-of-funnel leak is partially definitional, not behavioural. The *proportional* worst leak (view_item -> add_to_cart) is immune to this since both sides of the ratio are behavioural events.
7. **Sparse early window for `add_to_cart`.** The Nov 1-15 `add_to_cart` row is thin (<5 events/day) in the public sample; treat the first fortnight's per-day CVR as noise and weight conclusions on the Nov 16 onward segment.

## 12 · Business Impact



**Finding — two complementary leaks.** On this window, two stages dominate the funnel:

| Transition | Drop-off rate | Users lost | Commentary |
|---|---:|---:|---|
| `session_start` -> `view_item` | 77% | ~242K | Largest **absolute** cohort lost; partly definitional (bounces). |
| `view_item` -> `add_to_cart` | 80% | ~58K | Largest **proportional** leak; fully behavioural. |

The Session-2 A/B framework will prioritise interventions that lift the *absolute* leak (the `session_start -> view_item` funnel), because that is where marginal budget buys the most incremental activations — but will design each read against the *proportional* leak's cohort (`view_item -> add_to_cart`), because the tighter conversion rate converges to statistical power in fewer sessions.

**Translation to portfolio scale.** Using a portfolio-scale baseline (250K monthly SFP registrants, IDR 450K LTV per activated user), any intervention that shifts **overall entry-to-purchase CVR** by the percentages below is worth:

| Lift in overall CVR | Incremental activations / month | Annualised incremental CLV |
|---|---:|---:|
| **+0.5pp** | 1,250 | **~IDR 6.8B** |
| **+1pp** | 2,500 | **~IDR 13.5B** |
| **+2pp** | 5,000 | **~IDR 27B** (README hero) |
| **+3pp** | 7,500 | **~IDR 40.5B** |

A +2pp overall lift — achievable by moving either of the two dominant leaks by roughly a third — is worth an eight-figure USD equivalent per year at portfolio scale.

**What this justifies.** The A/B framework notebook builds the frequentist + Bayesian A/B framework that converts "target the worst-leaking stage" from a judgment call into a pre-registered experiment with quantified posterior loss and a stopping rule. The funnel above tells us *where*; the framework tells us *how much we can trust the result* before staking ad spend on it.

**What could be wrong.**
1. The GA4 e-comm funnel is not a regulated tobacco funnel — age-gate and excise-stamp friction may dominate user drop-off in ways this sample can't show. The 1-3pp ceiling may be optimistic under regulation.
2. `session_start` over-counts bounces (see §11.6) — the *absolute* worst-leak number is inflated, though the proportional ranking stands.
3. Seasonality inflates Nov 2020 - Jan 2021. A production baseline should use a 12-month window with YoY smoothing.

These are the three interview questions I expect to field first, and each has a clean answer in the Session-2 roadmap.